In [ ]:
import pandas as pd
import sys
import os

if "Desktop" in os.getcwd():
    base_path = "."
else:
    base_path = "."

print(base_path)
sys.path.append(base_path)
    
from src.utils.path_utils import get_project_root
from src.preprocessing.consts import Datasets
from src.preprocessing.read_raw_data import read_raw_data
from src.weak_labeling.content_based.threshold_selection.threshold_selection import select_thresholds_for_content_features
from src.weak_labeling.consts.content_features import ContentFeatures

covid_df: pd.DataFrame = read_raw_data(Datasets.INJECTED_NELA_COVID_DATASET_SAMPLED)
# elections_df: pd.DataFrame = read_raw_data(Datasets.NELA_ELECTIONS_DATASET_SAMPLED)

In [ ]:
covid_df.head()

In [ ]:

for feature in ContentFeatures.get_all():
    print(getattr(ContentFeatures, feature))
# for feature in vars(ContentFeatures):
#     if "__" not in feature:
#         print(feature)

In [ ]:
real_desc, fake_desc = select_thresholds_for_content_features(covid_df)

In [ ]:
wps_desc_df = pd.concat(
    [real_desc['content_words_per_sentence'], fake_desc['content_words_per_sentence']], 
    axis=1, 
    keys=['real', 'fake']
)

In [ ]:
wps_desc_df["abs_diff"] = wps_desc_df["real"] - wps_desc_df["fake"]

In [ ]:
wps_desc_df.dtypes

In [ ]:
wps_desc_df

In [ ]:
def is_string_series(s : pd.Series):
    if isinstance(s.dtype, pd.StringDtype):
        # The series was explicitly created as a string series (Pandas>=1.0.0)
        return True
    elif s.dtype == 'object':
        # Object series, check each value
        return all((v is None) or isinstance(v, str) for v in s)
    else:
        return False

In [ ]:
print(is_string_series(covid_df.content))

In [ ]:
print(len(covid_df))
covid_df.dropna(inplace=True)
print(len(covid_df))

In [ ]:
for idx, row in covid_df.iterrows():
    if not isinstance(row.title, str):
        print(row.title)
        break

In [ ]:
print(covid_df.dtypes)
print(len(covid_df))

In [ ]:
from src.utils.preprocessing_utils import cleanup_df

print(len(covid_df))
covid_cleaned_df = cleanup_df(covid_df)
print(len(covid_cleaned_df))

In [ ]:
covid_cleaned_df['content_length']  = covid_cleaned_df['content'].str.len()

In [ ]:
covid_cleaned_df["content"].apply(len).mean()

In [ ]:
covid_cleaned_df.head()

In [ ]:
index = covid_cleaned_df.content_length.argmax()
covid_cleaned_df.iloc[index]

In [ ]:
covid_df_shorter = covid_cleaned_df[covid_cleaned_df['content'].apply(len) < 5000] 

In [ ]:
len(covid_df_shorter)

In [ ]:
covid_df_shorter.hist(column='content_length', bins=200);

In [ ]:
import en_core_web_sm
nlp = en_core_web_sm.load()

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")

for idx, row in covid_df_shorter.iterrows():
    doc = nlp(row.content)
    for token in doc:
        print(token.pos_)
    break